In [0]:
from pyspark.sql import functions as F

df = spark.table("era5_project.silver.era5_clean")

# Map: select the required fields
mapped_df = (
    df.select(
        "pressure_level",
        "temperature_c",
        "geopotential_height_m"
    )
    .filter(
        F.col("pressure_level").isNotNull() &
        F.col("temperature_c").isNotNull() &
        F.col("geopotential_height_m").isNotNull()
    )
)

# Reduce: aggregate statistics by pressure level
result_df = (
    mapped_df
    .groupBy("pressure_level")
    .agg(
        F.sum("temperature_c").alias("total_temperature_c"),
        F.count("temperature_c").alias("observation_count"),
        F.avg("temperature_c").alias("average_temperature_c"),
        F.avg("geopotential_height_m").alias("average_geopotential_height_m")
    )
    .withColumn(
        "average_geopotential_height_km",
        F.col("average_geopotential_height_m") / 1000
    )
    .orderBy("pressure_level")
)

display(result_df)

pressure_level,total_temperature_c,observation_count,average_temperature_c,average_geopotential_height_m,average_geopotential_height_km
50,-2303623.7109400043,39076,-58.952393053024984,20645.520110636,20.645520110636
100,-2393102.106369977,39076,-61.24224860195457,16337.437039576276,16.337437039576276
150,-2215129.058930003,39076,-56.68771263512138,13798.542268127176,13.798542268127175
200,-2100633.3220999897,39076,-53.75763440730857,11962.382609170143,11.962382609170144
250,-1949620.8239500008,39076,-49.89305005502101,10519.18796058809,10.51918796058809
300,-1678160.764620004,39076,-42.94607341130116,9310.411384240242,9.310411384240242
400,-1112229.7837000107,39076,-28.463245565052993,7312.0681607042925,7.3120681607042926
500,-653571.1394199992,39076,-16.725640787695752,5674.831290306374,5.674831290306375
600,-293170.23354999686,39076,-7.502565092383992,4280.6786911701965,4.280678691170197
700,-14026.851839999135,39076,-0.35896334937043545,3064.419306086544,3.064419306086544


In [0]:
from pyspark.sql import functions as F

df = spark.table("era5_project.silver.era5_clean").select(
    "valid_time",
    "geopotential_height_m",
    "temperature_c",
    "relative_humidity",
    "pressure_level"
)

current_df = df.withColumn(
    "target_time",
    F.expr("valid_time + INTERVAL 24 HOURS")
)

future_df = df.select(
    F.col("valid_time").alias("future_time"),
    F.col("pressure_level").alias("future_pressure_level"),
    F.col("temperature_c").alias("target_temperature_c")
)

ml_df = (
    current_df.join(
        future_df,
        (current_df["target_time"] == future_df["future_time"]) &
        (current_df["pressure_level"] == future_df["future_pressure_level"]),
        "inner"
    )
    .select(
        current_df["valid_time"],
        current_df["target_time"],
        current_df["geopotential_height_m"],
        current_df["temperature_c"],
        current_df["relative_humidity"],
        current_df["pressure_level"],
        future_df["target_temperature_c"]
    )
)

print("Rows available for ML:", ml_df.count())
display(ml_df.limit(10))

Rows available for ML: 507936


valid_time,target_time,geopotential_height_m,temperature_c,relative_humidity,pressure_level,target_temperature_c
2000-01-01T12:00:00.000Z,2000-01-02T12:00:00.000Z,3045.4422254286633,1.530020000000036,36.79726,700,3.231840000000034
2000-01-01T12:00:00.000Z,2000-01-02T12:00:00.000Z,9377.137962505036,-39.46568999999997,14.784679,300,-38.47934999999998
2000-01-01T12:00:00.000Z,2000-01-02T12:00:00.000Z,13928.55256382149,-56.38636999999997,4.848752,150,-57.810479999999984
2000-01-02T12:00:00.000Z,2000-01-03T12:00:00.000Z,70.53859370937069,16.309600000000046,88.86223,1000,15.436360000000036
2000-01-02T12:00:00.000Z,2000-01-03T12:00:00.000Z,728.6095455634697,11.36474000000004,89.19455,925,9.379750000000001
2000-01-02T18:00:00.000Z,2000-01-03T18:00:00.000Z,16388.19678483478,-63.60759999999999,6.1574097,100,-61.69456999999997
2000-01-03T18:00:00.000Z,2000-01-04T18:00:00.000Z,13802.502383586649,-52.89236999999997,2.1437387,150,-56.545229999999975
2000-01-04T06:00:00.000Z,2000-01-05T06:00:00.000Z,2975.2136560395243,-2.9251499999999737,37.93555,700,2.036580000000015
2000-01-04T12:00:00.000Z,2000-01-05T12:00:00.000Z,16362.748747023705,-63.30854999999997,6.189025,100,-63.72786999999997
2000-01-05T00:00:00.000Z,2000-01-06T00:00:00.000Z,10541.838956218486,-51.362419999999986,37.95856,250,-51.65229999999997


In [0]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "geopotential_height_m",
    "temperature_c",
    "relative_humidity",
    "pressure_level"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

model_df = assembler.transform(ml_df).select(
    "valid_time",
    "target_time",
    "features",
    "target_temperature_c"
)

display(model_df.limit(10))

valid_time,target_time,features,target_temperature_c
2000-01-01T12:00:00.000Z,2000-01-02T12:00:00.000Z,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""3045.4422254286633"",""1.530020000000036"",""36.79726"",""700.0""]}",3.231840000000034
2000-01-01T12:00:00.000Z,2000-01-02T12:00:00.000Z,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""9377.137962505036"",""-39.46568999999997"",""14.784679"",""300.0""]}",-38.47934999999998
2000-01-01T12:00:00.000Z,2000-01-02T12:00:00.000Z,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""13928.55256382149"",""-56.38636999999997"",""4.848752"",""150.0""]}",-57.810479999999984
2000-01-02T12:00:00.000Z,2000-01-03T12:00:00.000Z,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""70.53859370937069"",""16.309600000000046"",""88.86223"",""1000.0""]}",15.436360000000036
2000-01-02T12:00:00.000Z,2000-01-03T12:00:00.000Z,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""728.6095455634697"",""11.36474000000004"",""89.19455"",""925.0""]}",9.379750000000001
2000-01-02T18:00:00.000Z,2000-01-03T18:00:00.000Z,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""16388.19678483478"",""-63.60759999999999"",""6.1574097"",""100.0""]}",-61.69456999999997
2000-01-03T18:00:00.000Z,2000-01-04T18:00:00.000Z,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""13802.502383586649"",""-52.89236999999997"",""2.1437387"",""150.0""]}",-56.545229999999975
2000-01-04T06:00:00.000Z,2000-01-05T06:00:00.000Z,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2975.2136560395243"",""-2.9251499999999737"",""37.93555"",""700.0""]}",2.036580000000015
2000-01-04T12:00:00.000Z,2000-01-05T12:00:00.000Z,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""16362.748747023705"",""-63.30854999999997"",""6.189025"",""100.0""]}",-63.72786999999997
2000-01-05T00:00:00.000Z,2000-01-06T00:00:00.000Z,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""10541.838956218486"",""-51.362419999999986"",""37.95856"",""250.0""]}",-51.65229999999997


In [0]:
# Calculate the chronological cutoff at 80% of the time range
time_range = ml_df.agg(
    F.min("valid_time").alias("min_time"),
    F.max("valid_time").alias("max_time")
).first()

min_time = time_range["min_time"]
max_time = time_range["max_time"]

cutoff_seconds = (
    max_time.timestamp() - min_time.timestamp()
) * 0.8

from datetime import timedelta

cutoff_time = min_time + timedelta(seconds=cutoff_seconds)

# Training targets must occur before the cutoff
train_df = model_df.filter(
    F.col("target_time") < F.lit(cutoff_time)
)

# Test observations begin at or after the cutoff
test_df = model_df.filter(
    F.col("valid_time") >= F.lit(cutoff_time)
)

print("Cutoff time:", cutoff_time)
print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Cutoff time: 2021-05-24 04:48:00
Training rows: 406289
Testing rows: 101595


In [0]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="target_temperature_c",
    predictionCol="prediction",
    maxDepth=10,
    minInstancesPerNode=50,
    seed=42
)

dt_model = dt.fit(train_df)

predictions = dt_model.transform(test_df)

display(
    predictions.select(
        "valid_time",
        "target_time",
        "target_temperature_c",
        "prediction"
    )
)

valid_time,target_time,target_temperature_c,prediction
2021-05-25T18:00:00.000Z,2021-05-26T18:00:00.000Z,9.751300000000015,9.845762307692338
2021-05-26T00:00:00.000Z,2021-05-27T00:00:00.000Z,7.070340000000044,7.601225671091472
2021-05-27T18:00:00.000Z,2021-05-28T18:00:00.000Z,-55.33995999999999,-55.23577565259577
2021-05-28T00:00:00.000Z,2021-05-29T00:00:00.000Z,-3.6065999999999576,-3.2913494466936353
2021-05-29T06:00:00.000Z,2021-05-30T06:00:00.000Z,-48.09362999999999,-50.994133859275
2021-05-30T18:00:00.000Z,2021-05-31T18:00:00.000Z,8.48397,7.929816609485388
2021-05-31T12:00:00.000Z,2021-06-01T12:00:00.000Z,-4.0811199999999985,-4.831442010050229
2021-06-01T12:00:00.000Z,2021-06-02T12:00:00.000Z,7.853420000000028,9.015915013960932
2021-06-02T18:00:00.000Z,2021-06-03T18:00:00.000Z,-59.68567999999999,-57.78261200881232
2021-06-03T12:00:00.000Z,2021-06-04T12:00:00.000Z,-58.03413999999998,-54.43975935742965


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

mae_evaluator = RegressionEvaluator(
    labelCol="target_temperature_c",
    predictionCol="prediction",
    metricName="mae"
)

rmse_evaluator = RegressionEvaluator(
    labelCol="target_temperature_c",
    predictionCol="prediction",
    metricName="rmse"
)

r2_evaluator = RegressionEvaluator(
    labelCol="target_temperature_c",
    predictionCol="prediction",
    metricName="r2"
)

mae = mae_evaluator.evaluate(predictions)
rmse = rmse_evaluator.evaluate(predictions)
r2 = r2_evaluator.evaluate(predictions)

print(f"MAE:  {mae:.4f} °C")
print(f"RMSE: {rmse:.4f} °C")
print(f"R²:   {r2:.4f}")

MAE:  1.9564 °C
RMSE: 2.6012 °C
R²:   0.9914


In [0]:
display(
    predictions.select(
        "target_temperature_c",
        "prediction"
    )
)

target_temperature_c,prediction
9.751300000000015,9.845762307692338
7.070340000000044,7.601225671091472
-55.33995999999999,-55.23577565259577
-3.6065999999999576,-3.2913494466936353
-48.09362999999999,-50.994133859275
8.48397,7.929816609485388
-4.0811199999999985,-4.831442010050229
7.853420000000028,9.015915013960932
-59.68567999999999,-57.78261200881232
-58.03413999999998,-54.43975935742965


Databricks visualization. Run in Databricks to view.